### Description

This notebook contains the implementation of fine-tuning a DeepSeek-R1 model to be able to reverse given words more accurately.

DeepSeek seems to reverse some typical words quite okay, although it is quite prone to more errors. But when giving the model sequences such as 'bananana' or 'monkekey', it is not able to reverse the letters in the sequences correctly, but returns something like 'ananabnan' or 'yekkenom'.

#### Implementation

In [ ]:
import os

# This ensures that cache dir path is not too long on WINDOWS. Will pop up an annoying error when running train().
# should be able to be removed when using unix-based systems.
# Comment it out if not using windows
os.environ["TRITON_CACHE_DIR"] = r"C:\triton_cache"

from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments 
import json
from datasets import Dataset
from transformers.trainer_utils import set_seed

#### Options

In [ ]:
train_model = False # if False, script will load a local model
train_full_epoch = True

max_seq_length = 2048
random_seed = 42

set_seed(random_seed)

test_words = ["tampere", "oulu", "saippuakauppias"]

#### Load model

Loads the pre-trained DeepSeek-R1 model.

In [ ]:
pt_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/DeepSeek-R1-Distill-Llama-8B',
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True
)

EOS_TOKEN = tokenizer.eos_token

#### Pre-trained Generation

Defines prompt style to be used for generation. Loops over the test samples to produce the reversed predictions and CoTs.

In [ ]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You should reverse the word(s) given by the user.

### Question:
{}

### Response:
<think>{}"""

In [ ]:
FastLanguageModel.for_inference(pt_model) 

for i, test_word in enumerate(test_words):
    question = "Reverse the word '{}'.".format(test_word)

    inputs = tokenizer(
        [prompt_style.format(question, "")],
        return_tensors="pt"
    ).to("cuda")
    
    outputs = pt_model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1200,
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs)[0].split('### Response:')[1]
    response = response.replace(EOS_TOKEN, "<EOS>")

    print(f"*** Test word {i+1} ***\n{response}\n\n")

#### Dataset

Load dataset generated by generate_data.py. The data is stored in a .json file, which is convenient to load into a Dataset.

In [ ]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You should reverse the word(s) given by the user.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

In [ ]:
path = "data/reverse_dataset.json"

with open(path, "r") as f:
    d = json.load(f)
    dataset = Dataset.from_list(d["data"])

dataset

### Model setup

Setup the model and training.

In [ ]:
model = FastLanguageModel.get_peft_model(
    pt_model,
    r=16, 
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=random_seed,
    loftq_config=None
)

In [ ]:
def define_prompts(samples):
    user_inputs = samples["input"]
    reasonings = samples["think"]
    outputs = samples["final"]

    prompts = []
    
    for x, y, z in zip(user_inputs, reasonings, outputs):
        prompt = train_prompt_style.format(x,y,z) + EOS_TOKEN
        prompts.append(prompt)

    return prompts

if train_full_epoch:
    train_args = TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs = 1,
        warmup_ratio=0.01,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=random_seed,
        output_dir="outputs"
    )
else:
    train_args = TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=random_seed,
        output_dir="outputs"
    )

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="prompt",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    formatting_func=define_prompts,
    args=train_args
)

#### Train the model

Either train the model if train_model=True. Otherwise, try to load an existing fine-tuned model from local files.

In [ ]:
if train_model:
    trainer_stats = trainer.train()
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "DeepSeek-R1-Word-Reverser",  # local folder
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = True
    )

#### Fine-tuned Generation

Predict the reversed words with the fine-tuned model.

In [ ]:
FastLanguageModel.for_inference(model) 

for i, test_word in enumerate(test_words):
    question = "Reverse the word '{}'.".format(test_word)

    inputs = tokenizer(
        [prompt_style.format(question, "")],
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1200,
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs)[0].split('### Response:')[1]
    response = response.replace(EOS_TOKEN, "<EOS>")
    
    print(f"*** Test word {i+1} ***\n{response}\n\n")

#### Save model

If a new model was trained, save it to local files.

In [ ]:
if train_model:
    local_model = "DeepSeek-R1-Word-Reverser"
    
    model.save_pretrained(local_model) 
    tokenizer.save_pretrained(local_model)
    model.save_pretrained_merged(local_model,
                                 tokenizer,
                                 save_method="merged_16bit")